In [1]:
import layers
# Здесь у нас находятся слои для 

import wmt_dataset
# Загрузчик датасетов с HugginFace. Внутри генерируются train/val/test + есть функция фильтрации и Коллатор

import train
# Там находится класс TransformerTrainer. Там все трениться, есть проверка на val датасете
# Вроде как понял, что до как радотает, но все еще поверхностно

import transformer
# Здесь находиться трансформер, состоящий из layers

from tokenizers import Tokenizer
device = 'cuda:0'
tokenizer = Tokenizer.from_file("mistral_tokenizer.json")
tokenizer.add_special_tokens(['<pad>', '<s>', '</s>'])

3

# 1. Архитектура модели
Создайте класс GeneratorTransformer, который авторегрессивно генерирует продолжение текста. Обучите его на книгах или каких-нибудь текстах, которые вы найдете в интернете


In [3]:
import torch
import math
from copy import deepcopy
from typing import Union, Optional, List, Tuple, Callable, Any
import os
    

class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000) -> None:
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(2 * torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.pe = torch.zeros(max_len, d_model)
        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = self.pe.unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.cpu()
        x = x + self.pe[:, :x.size(1), :]
        return x.to(device)

class Embedding(torch.nn.Module):
    def __init__(self, d_model: int, vocab_len: int, pad_index: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.embedding = torch.nn.Embedding(vocab_len, self.d_model, padding_idx=pad_index)
        self.positional_encoding = PositionalEncoding(self.d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)
        x = self.positional_encoding(x)
        return x
    
class ScaledDotProductAttention(torch.nn.Module):
    def __init__(self, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.factor = math.sqrt(self.d_model)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        scores = torch.matmul(query, key.transpose(-2, -1)) / self.factor
        if mask is not None:
            scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)
        attention_weights = torch.nn.Softmax(dim=-1)(scores)
        out = torch.matmul(attention_weights, value)
        return out, attention_weights
    
class MultiheadAttention(torch.nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.q_linear = torch.nn.Linear(d_model, d_model)
        self.k_linear = torch.nn.Linear(d_model, d_model)
        self.v_linear = torch.nn.Linear(d_model, d_model)
        self.out_linear = torch.nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(d_model)

        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_size = query.size(0)

        query = self.q_linear(query)
        query = query.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        key = self.k_linear(key)
        key = key.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        value = self.v_linear(value)
        value = value.view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        if mask is not None:
            mask = mask.unsqueeze(1)
        x, attention_weights = self.attention(query, key, value, mask)

        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        x = self.out_linear(x)
        x = self.dropout(x)
        return x, attention_weights
    
class FeedForward(torch.nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        if d_ff % d_model:
            raise ValueError(f"Feed forward dimension {d_ff} must be divisible by model dimension {d_model}")
        self.linear1 = torch.nn.Linear(d_model, d_ff)
        self.linear2 = torch.nn.Linear(d_ff, d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.nn.functional.relu(self.linear1(x))
        x = self.linear2(self.dropout(x))
        return x
    
class EncoderLayer(torch.nn.Module):
    def __init__(self, mha: MultiheadAttention, ffn: FeedForward, dropout: float = 0.1) -> None:
        super().__init__()
        self.attention = deepcopy(mha)
        self.ffn = deepcopy(ffn)
        self.layernorm1 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm2 = torch.nn.LayerNorm(mha.d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x_norm = self.layernorm1(x)
        x = x + self.attention(x_norm, x_norm, x_norm, mask)[0]
        x_norm = self.layernorm2(x)
        x = self.dropout(x + self.ffn(x_norm))
        return x
    
class DecoderLayer(torch.nn.Module):
    def __init__(self, mha: MultiheadAttention, enc_dec_mha: MultiheadAttention, ffn: FeedForward, dropout: float = 0.1) -> None:
        super().__init__()
        self.attention1 = deepcopy(mha)
        self.attention2 = deepcopy(enc_dec_mha)
        self.ffn = deepcopy(ffn)
        self.layernorm1 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm2 = torch.nn.LayerNorm(mha.d_model)
        self.layernorm3 = torch.nn.LayerNorm(mha.d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_memory: torch.Tensor, src_mask: Optional[torch.Tensor], tgt_mask: Optional[torch.Tensor]) -> torch.Tensor:
        x_norm = self.layernorm1(x)
        x = x + self.attention1(x_norm, x_norm, x_norm, tgt_mask)[0]
        
        # Encoder-decoder attention с маской энкодера
        x_norm = self.layernorm2(x)
        x = x + self.attention2(x_norm, encoder_memory, encoder_memory, src_mask)[0]
        
        # Feed-forward с маской декодера
        x_norm = self.layernorm3(x)
        x = self.dropout(x + self.ffn(x_norm))
        return x
    
class Encoder(torch.nn.Module):
    def __init__(self, enc_layer: EncoderLayer, num_layers: int) -> None:
        super().__init__()
        self.layers = torch.nn.ModuleList([deepcopy(enc_layer) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return x

class Decoder(torch.nn.Module):
    def __init__(self, dec_layer: DecoderLayer, num_layers: int) -> None:
        super().__init__()
        self.layers = torch.nn.ModuleList([deepcopy(dec_layer) for _ in range(num_layers)])

    def forward(self, x: torch.Tensor, encoder_memory: torch.Tensor, src_mask: Optional[torch.Tensor], tgt_mask: Optional[torch.Tensor]) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_memory, src_mask, tgt_mask)
        return x

In [4]:
def get_pad_mask(x: torch.Tensor, pad_index: int):
    x = x.cpu()
    x = (x != pad_index).unsqueeze(-2)
    return x.to(device)

def get_subsequent_mask(x: torch.Tensor):
    x = x.cpu()
    batch_size, seq_len = x.size()
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
    mask = mask.unsqueeze(0).expand(batch_size, -1, -1)
    return mask.to(device)


class GeneratorTransformer(torch.nn.Module):
    def __init__(
        self,
        d_model: int = 256,
        num_heads: int = 8,
        d_ff: int = 512,
        num_layers: int = 6,
        vocab_size: int = 1000,
        pad_index: int = 1,
        dropout: float = 0.1,
        max_len: int = 64,
        tokenizer: Tokenizer = None,
        device: str = 'cuda:0'
    ):
        super().__init__()
        # Инициализация аналогична вашему Transformer
        mha = MultiheadAttention(d_model, num_heads)
        enc_dec_mha = MultiheadAttention(d_model, num_heads)
        ffn = FeedForward(d_model, d_ff)
        
        self.encoder = Encoder(EncoderLayer(mha, ffn, dropout), num_layers)
        self.decoder = Decoder(DecoderLayer(mha, enc_dec_mha, ffn, dropout), num_layers)
        self.normalize = torch.nn.LayerNorm(d_model)

        self.embedding = Embedding(d_model, vocab_size, pad_index)  # Общее embedding для encoder и decoder
        self.vocab_projection = torch.nn.Linear(d_model, vocab_size)

        self.pad_index = pad_index
        self.device = device
        self.max_len = max_len
        self.tokenizer = tokenizer
        self.d_model = d_model
        
    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        # Forward pass для обучения
        memory, src_mask = self.encode(src)
        output = self.decode(tgt, memory, src_mask)
        return output
    
    def encode(self, src: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        src_mask = get_pad_mask(src, self.pad_index)
        src_emb = self.embedding(src)
        memory = self.encoder(src_emb, src_mask)
        return memory, src_mask
    
    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        tgt_mask = get_pad_mask(tgt, self.pad_index) & get_subsequent_mask(tgt)
        tgt_emb = self.embedding(tgt)
        output = self.decoder(tgt_emb, memory, src_mask, tgt_mask)
        output = self.normalize(output)
        return self.vocab_projection(output)


        
    def generate(self, prompt, context_len=50, temperature=1.0, max_out_tokens=200):
        """
        Генерирует ответ на основе промпта.
        
        При авторегрессии контекст сдвигается на 1 токен влево:
        - Изначально: [prompt_tokens]
        - После первого предсказания: [prompt_tokens, predicted_token]
        - При следующем предсказании: [prompt_tokens[1:], predicted_token, new_prediction]
        - И так далее, пока не достигнем max_length или EOS
        """
        self.eval()
        with torch.no_grad():
            # Токенизируйте промпт
            input_ids = self.tokenizer.encode(prompt).ids
            input_ids = torch.tensor([input_ids]).to(self.device)
            
            generated = input_ids.clone()
            
            for _ in range(max_out_tokens):
            # Получите предсказание для последнего токена
                outputs = self(input_ids, input_ids)
                next_token_logits = outputs[0, -1, :] / temperature
                
                # Выберите следующий токен
                next_token = torch.multinomial(torch.softmax(next_token_logits, dim=-1), 1).unsqueeze(0) 
                print(next_token)
                
                # Добавьте к результату
                generated = torch.cat([generated, next_token], dim=1)
                
                # Сдвиньте контекст на 1 токен влево для следующей итерации
                input_ids = generated[:, -self.max_len:]
                
                # Проверьте на EOS
                if next_token.item() == self.tokenizer.token_to_id('</s>'):
                    break
        
        return self.tokenizer.decode(generated[0].tolist())

    
        result = []
        for token_id in generated[0].tolist():
            if token_id not in [eos_token_id, pad_token_id]:
                result.append(token_id)
        
        return self.tokenizer.decode(result)

    def _apply_generation_parameters(
        self,
        logits: torch.Tensor,
        generated_seq: torch.Tensor,
        temperature: float,
        top_k: int,
        top_p: float,
        repetition_penalty: float
    ) -> torch.Tensor:
        """Применяет параметры генерации к логарифмам"""
        # Температура
        if temperature != 1.0:
            logits = logits / temperature
            
        # Штраф за повторения
        if repetition_penalty != 1.0:
            for token_id in torch.unique(generated_seq):
                logits[:, token_id] = logits[:, token_id] / repetition_penalty
                
        # Top-k фильтрация
        if top_k > 0:
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = -float('Inf')
            
        # Top-p (nucleus) sampling
        if top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.nn.functional.softmax(sorted_logits, dim=-1), dim=-1)
            
            # Удаляем токены с cumulative probability выше threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Сдвигаем на 1, чтобы сохранить первый токен выше threshold
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            
            for i in range(logits.size(0)):
                indices_to_remove = sorted_indices[i][sorted_indices_to_remove[i]]
                logits[i][indices_to_remove] = -float('Inf')
                
        return logits

# 2. Токенизация
Используйте тот же токенизатор, что и в уроке, или создайте более простой:

In [6]:
tokenizer.get_vocab_size()

32003

# 4. Обучение
Рекомендуемые параметры:
```
batch_size = 1 (для экономии памяти)
max_length = 128-192 (размер контекста)
learning_rate = 1e-4
num_epochs = 2-4
```
Помните, что модель должна просмотреть ВЕСЬ ваш текст. Сдвигайте "окно" по тексту на max_length и итерируйтесь по всему тексту (подумайте, как создать датасет для этого). Варьируйте контекст, разделяя их на законченные блоки (абзацы, предложения), не забывайте про eos и bos токены!


In [8]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        with open(file_path, encoding='utf8') as f:
            self.texts = [line.strip() for line in f if line.strip()]
        
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        bos_token_id = self.tokenizer.token_to_id('<s>')
        eos_token_id = self.tokenizer.token_to_id('</s>')
        pad_token_id = self.tokenizer.token_to_id('<pad>')

        
        tokens = [bos_token_id] + self.tokenizer.encode(text).ids + [eos_token_id]
        tokens = torch.tensor(tokens[:self.max_length])
        return tokens
        



class Collator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, batch):
        # batch - список тензоров разной длины
        # Применяем паддинг ко всем последовательностям в батче
        padded_batch = pad_sequence(
            batch,
            batch_first=True,
            padding_value=self.pad_token_id
        )
        return padded_batch
        
# Датасет и загрузчик
dataset = TextDataset(r"japan.txt_Ascii.txt", tokenizer)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [9]:
torch.cuda.empty_cache()

In [10]:
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import torch.nn as nn
vocab_size = tokenizer.get_vocab_size()
device = 'cuda:0'

colate_func = Collator(tokenizer.token_to_id('<pad>'))
dataset = TextDataset(r"japan.txt_Ascii.txt", tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=colate_func)

model = GeneratorTransformer(
    d_model=512,
    num_heads=8,
    vocab_size=vocab_size,
    pad_index=tokenizer.token_to_id('<pad>'),
    tokenizer=tokenizer,
    device=device,
    max_len=96).to(device)

# Обучение
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id('<pad>'))
scaler = GradScaler(device=device)  # Инициализация один раз перед циклом обучения

for epoch in tqdm(range(8)):
    model.train()
    total_loss = 0
    
    for idx, batch in enumerate(dataloader):
        batch = batch.to(device)
        
        # Сдвиг вправо для teacher forcing
        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        optimizer.zero_grad()
        
        with autocast(device_type=device, dtype=torch.float16):
        
            outputs = model(inputs, targets)
            loss = loss_fn(
                outputs.view(-1, vocab_size),
                targets.reshape(-1))
            scaler.scale(loss).backward()
            

        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch}, Loss: {avg_loss:.4f}")

 12%|██████████▌                                                                         | 1/8 [00:47<05:32, 47.56s/it]

Epoch 0, Loss: 4.4392


 25%|█████████████████████                                                               | 2/8 [01:36<04:49, 48.29s/it]

Epoch 1, Loss: 0.8625


 38%|███████████████████████████████▌                                                    | 3/8 [02:26<04:05, 49.12s/it]

...

Epoch 5, Loss: 0.0421


 88%|█████████████████████████████████████████████████████████████████████████▌          | 7/8 [05:49<00:50, 50.65s/it]

Epoch 6, Loss: 0.0278


100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [06:41<00:00, 50.20s/it]

Epoch 7, Loss: 0.0195


In [26]:
def chat():
    model.eval()
    
    while True:
        user_input = input("Вы: ")
        if user_input.lower() == 'quit':
            break
            
        response = model.generate(user_input, max_out_tokens=200, temperature=0.8)
        print(f"Бот: {response}")



In [28]:
chat()

Вы:  Сегодня я сделал


tensor([[[-4.2834, -3.7329, -0.3788,  ..., -4.7632, -3.7365, -2.9521],
         [-4.7140, -3.4211,  3.4761,  ..., -3.5328, -2.6879, -2.8019],
         [-4.2515, -3.1236, -0.2543,  ..., -4.2496, -3.2647, -2.6745],
         ...,
         [-3.4563, -3.6538, -0.2329,  ..., -3.9833, -3.2088, -2.7884],
         [-4.4871, -3.8274,  1.4504,  ..., -3.8290, -4.1881, -2.9714],
         [-3.7227, -4.1004,  0.6937,  ..., -3.2814, -3.5819, -3.7598]]],
       device='cuda:0')
tensor([[28794]], device='cuda:0')
tensor([[[-4.1984, -3.6442, -0.4004,  ..., -4.7594, -3.6070, -2.9456],
         [-4.6034, -3.3001,  3.5198,  ..., -3.4874, -2.5129, -2.7597],
         [-4.2094, -3.0398, -0.2086,  ..., -4.2706, -3.1229, -2.6697],
         ...,
         [-4.4397, -3.7023,  1.3988,  ..., -3.7797, -4.0541, -2.9336],
         [-3.6627, -3.9886,  0.6803,  ..., -3.3453, -3.4833, -3.7417],
         [-3.6423, -4.0509,  0.7722,  ..., -3.3752, -3.4986, -3.7019]]],
       device='cuda:0')
tensor([[28794]], device='cuda:0'

Вы:  Сегодня я прогулялся на вечерней улице и увидел, как с неба падают  


tensor([[[-3.9682, -3.3900, -0.4200,  ..., -4.1137, -3.4138, -2.7113],
         [-4.6468, -3.6339,  3.4405,  ..., -3.4136, -2.6000, -2.7385],
         [-4.3639, -3.4520,  0.1855,  ..., -4.3311, -3.2935, -2.8544],
         ...,
         [-4.4932, -4.8743,  0.1087,  ..., -2.8190, -3.6419, -3.5938],
         [-4.0887, -4.5002,  2.6152,  ..., -4.4900, -3.3496, -3.5720],
         [-2.9323, -3.6666, -0.2701,  ..., -3.3448, -3.4279, -2.9099]]],
       device='cuda:0')
tensor([[10185]], device='cuda:0')
tensor([[[-3.9576, -3.3842, -0.4100,  ..., -4.0860, -3.4328, -2.7015],
         [-4.6243, -3.6342,  3.4390,  ..., -3.3795, -2.6119, -2.7195],
         [-4.3446, -3.4207,  0.1947,  ..., -4.3093, -3.3019, -2.8413],
         ...,
         [-4.0750, -4.4807,  2.6036,  ..., -4.4607, -3.3819, -3.5643],
         [-2.9180, -3.6455, -0.2966,  ..., -3.3216, -3.4466, -2.9249],
         [-4.0210, -3.7754, -1.6575,  ..., -3.6811, -3.4719, -3.7579]]],
       device='cuda:0')
tensor([[10185]], device='cuda:0'

Вы:  Модель получилась очень плохая, но зато я получил опыт в написании и понимании трансформеров(хотя и не полностью разобрался)


tensor([[[-3.2882, -3.5836, -1.2218,  ..., -3.6641, -3.3755, -3.7159],
         [-3.0191, -4.0041, -2.1159,  ..., -2.9909, -3.8572, -3.5664],
         [-4.2725, -4.2980,  1.0833,  ..., -4.7208, -3.0495, -3.2187],
         ...,
         [-4.4675, -4.6054,  0.9423,  ..., -4.7279, -3.0448, -3.1385],
         [-3.2071, -3.7472,  1.7534,  ..., -3.5466, -3.7569, -3.9068],
         [-3.1824, -3.7947,  0.3672,  ..., -3.6770, -3.2186, -3.2971]]],
       device='cuda:0')
tensor([[28731]], device='cuda:0')
tensor([[[-3.2920, -3.5767, -1.2382,  ..., -3.6515, -3.3801, -3.7200],
         [-3.0359, -4.0033, -2.1498,  ..., -2.9716, -3.8620, -3.5809],
         [-4.2780, -4.2881,  1.0847,  ..., -4.7152, -3.0477, -3.2222],
         ...,
         [-3.2174, -3.7284,  1.7316,  ..., -3.5279, -3.7634, -3.9083],
         [-3.1825, -3.7863,  0.3469,  ..., -3.6679, -3.2236, -3.3040],
         [-3.1526, -3.7651,  0.3216,  ..., -3.7707, -3.2416, -3.2356]]],
       device='cuda:0')
tensor([[28731]], device='cuda:0'

Вы:  quit
